# Document Scanner — Kaggle GPU Corner Detection Training Launcher

Runs the corner detection comparison between Approach A and Approach B (`[REQ-30]`, `[REQ-31]`, ADR-007) on a Kaggle GPU (T4 / P100):

| Run | Model | Formulation | Loss | Target |
|---|---|---|---|---|
| exp-009 | `CornerRegNet` | Approach A: Direct Coordinate Regression | L1 | 8 normalized coordinates in [0, 1] |
| exp-010 | `CornerHeatmapNet` | Approach B: Heatmap Regression | MSE | 4-channel 512x512 Gaussians (sigma=8) |

### Kaggle GPU Advantage
- Kaggle provides **30 hours/week** of free GPU compute (NVIDIA P100 / T4 x2).
- Persistent output directory at `/kaggle/working/` allowing direct zip download of checkpoints and figures.

### Step 0: Confirm GPU & Environment Setup

In [ ]:
import os, sys, torch
print('PyTorch version:', torch.__version__)
print('vCPUs:', os.cpu_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print('WARNING: CUDA GPU not detected. Ensure Accelerator is set to GPU T4 x2 or P100 in Kaggle settings.')

### Step 1: Clone Repository & Change Working Directory

In [ ]:
import os
repo_dir = '/kaggle/working/DocEn'
if not os.path.exists(repo_dir):
    !git clone https://github.com/HedieTahmouresi/DocEn.git {repo_dir}
%cd {repo_dir}
!git pull
!git log --oneline -1

### Step 2: Install Dependencies & Extract Data

Note: If you attached `data.zip` as a Kaggle Dataset, this cell automatically checks `/kaggle/input/`.

In [ ]:
!pip install -q -r requirements.txt

import os, glob, shutil

if not os.path.exists('data/clean_scans'):
    # Search for data.zip in kaggle input directories
    zip_candidates = glob.glob('/kaggle/input/**/data.zip', recursive=True) + ['/kaggle/working/data.zip']
    if zip_candidates:
        src_zip = zip_candidates[0]
        print('Extracting data from:', src_zip)
        !unzip -q "{src_zip}" -d .
    else:
        # Check if unzipped dataset input exists
        clean_scans_candidates = glob.glob('/kaggle/input/**/clean_scans', recursive=True)
        if clean_scans_candidates:
            data_root = os.path.dirname(clean_scans_candidates[0])
            print('Found unzipped data at:', data_root)
            !cp -r "{data_root}"/* data/
        else:
            print('WARNING: data.zip not found in /kaggle/input/. Upload data.zip as a Kaggle Dataset and attach it.')

if not os.path.exists('data/frozen/val'):
    print('Frozen evaluation sets missing - generating them now...')
    !python -m src.data.freeze

### Step 3: Run Sanity Unit Tests

In [ ]:
!python -m pytest tests/test_corner_pipeline.py -v

### Step 4: Quick 1-Epoch Smoke Run

Verifies data loading, GPU forward/backward passes, and metric logging.

In [ ]:
!python train_corners.py --env colab_t4 --epochs 1 --samples-per-epoch 100 --allow-cpu-fallback

### Step 5: Full Paired Corner Detection Training (40 Epochs)

Executes paired training of `exp-009_corner_approach_a` (RegNet) and `exp-010_corner_approach_b` (HeatmapNet) side-by-side over 40 epochs.

In [ ]:
!python train_corners.py --env colab_t4

### Step 6: Evaluate Checkpoints & Generate Report

In [ ]:
!python -m scripts.evaluate_corners

### Step 7: Zip Output Results for Easy Download

Compresses `runs/` and `outputs/` into a downloadable zip file in `/kaggle/working/`.

In [ ]:
!zip -r /kaggle/working/corner_results.zip runs/ outputs/